# U2IE-Net on Google Colab

**Runtime → Change runtime type**
- Inference only: **T4 GPU** is enough.
- Training (paper: RTX 4090, ~13.5 h for 50 epochs): **A100 GPU** or **G4 GPU**, and turn on **High-RAM**.

U2IE-Net runs Retinex / CLAHE / Gaussian blur with OpenCV on the **CPU** inside every forward pass,
so training speed depends heavily on the number of vCPUs, not only on the GPU. Check `nproc` below.
Training may not finish in one Colab session: checkpoints go to Google Drive and `--resume` continues from the last finished epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!echo "vCPUs: $(nproc)" && free -h | head -2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything that must survive a disconnect lives here
DRIVE_ROOT = '/content/drive/MyDrive/U2IE_Net'
!mkdir -p "$DRIVE_ROOT/checkpoints"

In [ ]:
REPO_URL = 'https://github.com/nantapongpa-a11y/U2IE-Net.git'
BRANCH = 'U2IE-Net-Colab'

%cd /content
!rm -rf U2IE-Net && git clone -q -b $BRANCH $REPO_URL
%cd /content/U2IE-Net
!pip install -q kornia

## 1. Inference with the pre-trained weights

Download `u2ie_latest.pth` from the Baidu / Quark link in the README on your own computer,
then upload it to `MyDrive/U2IE_Net/checkpoints/` in Google Drive.

In [ ]:
CKPT = f'{DRIVE_ROOT}/checkpoints/u2ie_latest.pth'
INPUT_DIR = '/content/U2IE-Net/Inputs'      # or a folder on Drive
OUTPUT_DIR = f'{DRIVE_ROOT}/Outputs'

!python test.py --input_dir "$INPUT_DIR" --output_dir "$OUTPUT_DIR" --checkpoint "$CKPT" --batch_size 16

In [ ]:
# Side-by-side preview: input (top) vs enhanced (bottom)
import os, matplotlib.pyplot as plt
from PIL import Image

names = sorted(os.listdir(OUTPUT_DIR))[:6]
fig, ax = plt.subplots(2, len(names), figsize=(3 * len(names), 6))
for i, n in enumerate(names):
    ax[0, i].imshow(Image.open(os.path.join(INPUT_DIR, n)).resize((256, 256)))
    ax[1, i].imshow(Image.open(os.path.join(OUTPUT_DIR, n)))
    ax[0, i].set_title(n[:18], fontsize=8)
    for a in ax[:, i]: a.axis('off')
plt.tight_layout(); plt.show()

## 2. Training

Prepare the data the same way as the paper (Table 2) and upload it to Drive as one zip, `MyDrive/U2IE_Net/data.zip`:

```
data.zip
├── underwater/   # unpaired underwater images (UIEB-Raw + RUIE-UIQS + UICoD, 5000 images)
├── haze/         # RESIDE-β hazy images (5000)
└── clear/        # RESIDE-β clear images (5000)
```

`haze/` and `clear/` are paired **by sorted filename**, so the i-th file in `haze/` must match the i-th file in `clear/`
(rename them to the same names, e.g. `0001.jpg` in both folders).

The data is unzipped to the Colab local disk because reading thousands of small files straight from Drive is very slow.

In [ ]:
!mkdir -p /content/data && unzip -q -n "$DRIVE_ROOT/data.zip" -d /content/data
!for d in underwater haze clear; do echo "$$d: $$(ls /content/data/$$d | wc -l)"; done

In [ ]:
import os
NUM_WORKERS = max(1, os.cpu_count() - 1)

# Re-run this cell after a disconnect (after re-running the setup cells above): --resume continues from the last epoch
TRAIN_ARGS = ' '.join([
    '--u_path /content/data/underwater --h_path /content/data/haze --g_path /content/data/clear',
    f'--output_dir "{DRIVE_ROOT}/OpenData"',
    '--epochs 50 --batch_size 8 --lr 0.0005',
    f'--num_workers {NUM_WORKERS} --resume',
])
!python train.py {TRAIN_ARGS}

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage(f'{DRIVE_ROOT}/OpenData/lossdata/training_loss_curve.png'))